In [1]:
# =========================
# Cell 1) Imports
# =========================
from __future__ import annotations

import os
import re
import math
from dataclasses import dataclass
from datetime import datetime, date, time, timedelta
from zoneinfo import ZoneInfo
from typing import List, Dict, Optional, Any, Tuple

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

KST = ZoneInfo("Asia/Seoul")

In [2]:
# =========================
# Cell 2) Parameters & DB  (FIXED: ensure engine is defined)
# =========================
import os
from sqlalchemy import create_engine
from sqlalchemy.engine import Engine

PROD_DAY = "20260128"     # YYYYMMDD
SHIFT_TYPE = "day"        # "day" | "night"

WINDOW_SECONDS = 43200

STATIONS = ["FCT1", "FCT2", "FCT3", "FCT4", "Vision1", "Vision2"]
REMARKS  = ["PD", "Non-PD"]

IDEAL_BASE_PROD_DAY = "20260128"

DB_CONFIG = {
    "host": os.getenv("PGHOST", "100.105.75.47"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", ""),
}

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    eng = create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )
    return eng

# ✅ 이 줄이 실행되어야 Cell 5에서 engine을 쓸 수 있음
engine = make_engine()

# (선택) 연결 테스트: 바로 여기서 DB 접속 이상 여부 확인
try:
    with engine.begin() as conn:
        conn.exec_driver_sql("SELECT 1;")
    print("[DB] engine OK")
except Exception as e:
    print("[DB] engine FAILED:", repr(e))
    raise

[DB] engine OK


In [3]:
# =========================
# Cell 3) Utils: time parsing + window + interval (half-open)
# =========================

def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[:4]), int(s[4:6]), int(s[6:8]))

def _only_digits(s: str) -> str:
    return re.sub(r"[^0-9]", "", s)

def parse_hms_flexible(v: Any) -> time:
    """
    ✅ 유연 파서 (NameError 원인 해결: 여기서 정의)
    허용:
      - "HH:MI:SS"
      - "HH:MI"
      - "HH:MI:SS.xx" (소수초 제거)
      - "HHMMSS" / "HMMSS" / "HHMM"
      - time 객체
      - 숫자형(104000 등)
    """
    if v is None:
        raise ValueError("time value is None")

    if isinstance(v, time):
        return v

    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        raise ValueError(f"empty time value: {v}")

    # 소수초 제거
    if "." in s:
        s = s.split(".", 1)[0].strip()

    # 콜론 포맷
    if ":" in s:
        parts = s.split(":")
        if len(parts) == 2:
            hh, mm = parts
            ss = "0"
        else:
            hh, mm, ss = parts[0], parts[1], parts[2]
        return time(int(hh), int(mm), int(ss))

    # 숫자 포맷
    digits = _only_digits(s)
    if digits == "":
        raise ValueError(f"invalid time format: {v}")

    if len(digits) == 5:   # HMMSS
        digits = "0" + digits

    if len(digits) == 6:   # HHMMSS
        hh = int(digits[0:2])
        mm = int(digits[2:4])
        ss = int(digits[4:6])
        return time(hh, mm, ss)

    if len(digits) == 4:   # HHMM
        hh = int(digits[0:2])
        mm = int(digits[2:4])
        return time(hh, mm, 0)

    raise ValueError(f"unsupported time format: {v} (digits={digits})")

def build_shift_window(prod_day: str, shift_type: str) -> Tuple[datetime, datetime]:
    """
    ✅ half-open [start, end)
    day   = [08:30:00, 20:30:00)
    night = [20:30:00, 다음날 08:30:00)
    """
    d = yyyymmdd_to_date(prod_day)
    if shift_type == "day":
        start = datetime.combine(d, time(8, 30, 0), tzinfo=KST)
        end   = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
    elif shift_type == "night":
        start = datetime.combine(d, time(20, 30, 0), tzinfo=KST)
        end   = datetime.combine(d + timedelta(days=1), time(8, 30, 0), tzinfo=KST)
    else:
        raise ValueError("SHIFT_TYPE must be 'day' or 'night'")

    dur = int((end - start).total_seconds())
    if dur != WINDOW_SECONDS:
        raise ValueError(f"Window seconds mismatch: got {dur}, expected {WINDOW_SECONDS}")
    return start, end

WINDOW_START, WINDOW_END = build_shift_window(PROD_DAY, SHIFT_TYPE)

@dataclass(frozen=True)
class Interval:
    start: datetime
    end: datetime   # ✅ end EXCLUSIVE

def merge_intervals(intervals: List[Interval]) -> List[Interval]:
    """half-open에서 겹치거나 맞닿는(cur.start <= last.end) 구간 합침"""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x.start)
    merged = [intervals[0]]
    for cur in intervals[1:]:
        last = merged[-1]
        if cur.start <= last.end:
            merged[-1] = Interval(last.start, max(last.end, cur.end))
        else:
            merged.append(cur)
    # 0길이 제거
    return [iv for iv in merged if iv.end > iv.start]

def overlap_seconds(a: Interval, b: Interval) -> int:
    """half-open overlap"""
    s = max(a.start, b.start)
    e = min(a.end, b.end)
    if e <= s:
        return 0
    return int((e - s).total_seconds())

def total_seconds(intervals: List[Interval]) -> int:
    return sum(int((iv.end - iv.start).total_seconds()) for iv in intervals)

WINDOW_START, WINDOW_END

(datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')),
 datetime.datetime(2026, 1, 28, 20, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))

In [4]:
# =========================
# Cell 4) Table Names  (UPDATED)
# =========================
SAVE_SCHEMA = "i_daily_report"

T_REMARK_CHANGE_DAY   = "j_remark_change_day_daily"
T_REMARK_CHANGE_NIGHT = "j_remark_change_night_daily"

T_PLANNED_STOP_DAY    = "i_planned_stop_time_day_daily"
T_PLANNED_STOP_NIGHT  = "i_planned_stop_time_night_daily"

T_NON_TIME_DAY        = "i_non_time_day_daily"
T_NON_TIME_NIGHT      = "i_non_time_night_daily"

T_QUALITY_DAY         = "b_station_day_daily_percentage"
T_QUALITY_NIGHT       = "b_station_night_daily_percentage"

# (기존) station별 pass 집계
T_FINAL_AMT_DAY       = "a_station_day_daily_final_amount"
T_FINAL_AMT_NIGHT     = "a_station_night_daily_final_amount"

# ✅ (추가) Performance Total Count용 (remark별 총합)
T_TOTALCOUNT_DAY      = "a_day_daily_final_amount"
T_TOTALCOUNT_NIGHT    = "a_night_daily_final_amount"

# ✅ Ideal CT - Vision
IDEAL_SCHEMA_VISION = "e1_FCT_ct"
IDEAL_TABLE_VISION  = "fct_whole_op_ct"

# ✅ Ideal CT - FCT (요구사항)
IDEAL_SCHEMA_FCT = "e1_FCT_ct"     # (FCT 대문자 관련: 실제 스키마가 이 이름이면 그대로)
IDEAL_TABLE_FCT  = "fct_op_ct"

remark_change_table = T_REMARK_CHANGE_DAY if SHIFT_TYPE == "day" else T_REMARK_CHANGE_NIGHT
planned_stop_table  = T_PLANNED_STOP_DAY  if SHIFT_TYPE == "day" else T_PLANNED_STOP_NIGHT
non_time_table      = T_NON_TIME_DAY      if SHIFT_TYPE == "day" else T_NON_TIME_NIGHT
quality_table       = T_QUALITY_DAY       if SHIFT_TYPE == "day" else T_QUALITY_NIGHT

final_amt_table     = T_FINAL_AMT_DAY     if SHIFT_TYPE == "day" else T_FINAL_AMT_NIGHT
totalcount_table    = T_TOTALCOUNT_DAY    if SHIFT_TYPE == "day" else T_TOTALCOUNT_NIGHT

remark_change_fqn = f'"{SAVE_SCHEMA}"."{remark_change_table}"'
planned_stop_fqn  = f'"{SAVE_SCHEMA}"."{planned_stop_table}"'
non_time_fqn      = f'"{SAVE_SCHEMA}"."{non_time_table}"'
quality_fqn       = f'"{SAVE_SCHEMA}"."{quality_table}"'
final_amt_fqn     = f'"{SAVE_SCHEMA}"."{final_amt_table}"'
totalcount_fqn    = f'"{SAVE_SCHEMA}"."{totalcount_table}"'

ideal_vision_fqn   = f'"{IDEAL_SCHEMA_VISION}"."{IDEAL_TABLE_VISION}"'
ideal_fct_fqn      = f'"{IDEAL_SCHEMA_FCT}"."{IDEAL_TABLE_FCT}"'

remark_change_fqn, planned_stop_fqn, non_time_fqn, quality_fqn, final_amt_fqn, totalcount_fqn, ideal_vision_fqn, ideal_fct_fqn

('"i_daily_report"."j_remark_change_day_daily"',
 '"i_daily_report"."i_planned_stop_time_day_daily"',
 '"i_daily_report"."i_non_time_day_daily"',
 '"i_daily_report"."b_station_day_daily_percentage"',
 '"i_daily_report"."a_station_day_daily_final_amount"',
 '"i_daily_report"."a_day_daily_final_amount"',
 '"e1_FCT_ct"."fct_whole_op_ct"',
 '"e1_FCT_ct"."fct_op_ct"')

In [5]:
# =========================
# Cell 4.5) Helper: get_columns (ADD)
# =========================
from sqlalchemy import text
from sqlalchemy.engine import Engine
from typing import List

def get_columns(engine: Engine, schema: str, table: str) -> List[str]:
    sql = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name   = :table
        ORDER BY ordinal_position
    """)
    with engine.begin() as conn:
        rows = conn.execute(sql, {"schema": schema, "table": table}).fetchall()
    return [r[0] for r in rows]

In [6]:
# =========================
# Cell 5) Column Introspection (UPDATED)
# =========================
cols_remark = get_columns(engine, SAVE_SCHEMA, remark_change_table)
cols_stop   = get_columns(engine, SAVE_SCHEMA, planned_stop_table)
cols_non    = get_columns(engine, SAVE_SCHEMA, non_time_table)
cols_q      = get_columns(engine, SAVE_SCHEMA, quality_table)
cols_final  = get_columns(engine, SAVE_SCHEMA, final_amt_table)
cols_tc     = get_columns(engine, SAVE_SCHEMA, totalcount_table)

cols_ideal_vision = get_columns(engine, IDEAL_SCHEMA_VISION, IDEAL_TABLE_VISION)
cols_ideal_fct    = get_columns(engine, IDEAL_SCHEMA_FCT, IDEAL_TABLE_FCT)

cols_remark, cols_stop, cols_non[:20], cols_q[:20], cols_final[:15], cols_tc[:10], cols_ideal_vision[:20], cols_ideal_fct[:20]

(['prod_day',
  'station',
  'shift_type',
  'at_time',
  'from_remark',
  'to_remark',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'from_time',
  'to_time',
  'Total 계획 정지 시간',
  'total_planned_time',
  'updated_at'],
 ['prod_day',
  'shift_type',
  '비가동 FCT1',
  '비가동 FCT2',
  '비가동 Vision1',
  '비가동 FCT3',
  '비가동 FCT4',
  '비가동 Vision2',
  'Total Vision 비가동 시간',
  'vision1_non_time',
  'vision2_non_time',
  'total_vision_non_time',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'FCT1',
  'FCT2',
  'FCT3',
  'FCT4',
  'Vision1',
  'Vision2',
  'FCT>Vision Total',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'station',
  'pn',
  'remark',
  'A시간대(08:30:00 ~ 10:29:59)',
  'B시간대(10:30:00 ~ 12:29:59)',
  'C시간대(12:30:00 ~ 14:29:59)',
  'D시간대(14:30:00 ~ 16:29:59)',
  'E시간대(16:30:00 ~ 18:29:59)',
  'F시간대(18:30:00 ~ 20:29:59)',
  '합계',
  'updated_at'],
 ['prod_day',
  'shift_type',
  'pn',
  'remark',
  'A시간대(08:30:00 ~ 10:29:59)',
  'B시간대(10:30:00 ~ 12:29:59)',
  'C시간대(12:30:00 ~ 14:

In [7]:
# =========================
# Cell 6) Load Data (UPDATED: + totalcount)
# =========================
from sqlalchemy import text
import pandas as pd

def load_remark_changes() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {remark_change_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
        ORDER BY station, at_time
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "stations": list(STATIONS)
        })

def load_planned_stops() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {planned_stop_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE})

def load_non_time() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {non_time_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE})

def load_quality() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {quality_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={"prod_day": PROD_DAY, "shift_type": SHIFT_TYPE})

def load_final_amounts() -> pd.DataFrame:
    sql = text(f"""
        SELECT *
        FROM {final_amt_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND station = ANY(:stations)
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "stations": list(STATIONS)
        })

def load_totalcount_by_remark() -> pd.DataFrame:
    # ✅ remark별 "합계" 문자열에서 PASS+FAIL -> Total Count
    sql = text(f"""
        SELECT remark, "합계"
        FROM {totalcount_fqn}
        WHERE prod_day = :prod_day
          AND shift_type = :shift_type
          AND remark = ANY(:remarks)
    """)
    with engine.begin() as conn:
        return pd.read_sql(sql, conn, params={
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "remarks": list(REMARKS),
        })

df_remark = load_remark_changes()
df_stop   = load_planned_stops()
df_non    = load_non_time()
df_q      = load_quality()
df_final  = load_final_amounts()
df_tc     = load_totalcount_by_remark()

df_remark.head(), df_stop.head(), df_non.head(), df_q.head(), df_final.head(), df_tc

(   prod_day station shift_type   at_time from_remark to_remark  \
 0  20260128    FCT1        day  10:55:48      Non-PD        PD   
 1  20260128    FCT1        day  13:09:00          PD    Non-PD   
 2  20260128    FCT1        day  14:32:01      Non-PD        PD   
 3  20260128    FCT1        day  14:33:41          PD    Non-PD   
 4  20260128    FCT1        day  14:36:37      Non-PD        PD   
 
                         updated_at  
 0 2026-02-03 00:37:32.423999+00:00  
 1 2026-02-03 00:37:32.423999+00:00  
 2 2026-02-03 00:37:32.423999+00:00  
 3 2026-02-03 00:37:32.423999+00:00  
 4 2026-02-03 00:37:32.423999+00:00  ,
    prod_day shift_type from_time   to_time Total 계획 정지 시간  total_planned_time  \
 0  20260128        day                                10분                 600   
 1  20260128        day  10:50:00  11:00:00            10분                 600   
 
                  updated_at  
 0 2026-02-02 09:19:08+00:00  
 1 2026-02-02 09:19:08+00:00  ,
    prod_day shift_type  

In [8]:
# =========================
# Cell 7) PASS Parse & Total Column Detect (수정: FAIL/total 파서 추가)
# =========================
PASS_RE = re.compile(r"PASS\s*[:=]\s*(\d+)", re.IGNORECASE)
FAIL_RE = re.compile(r"FAIL\s*[:=]\s*(\d+)", re.IGNORECASE)

def parse_pass(x) -> int:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    m = PASS_RE.search(s)
    return int(m.group(1)) if m else 0

def parse_fail(x) -> int:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    m = FAIL_RE.search(s)
    return int(m.group(1)) if m else 0

def parse_total_count(x) -> int:
    # total = PASS + FAIL
    return parse_pass(x) + parse_fail(x)

def detect_total_col(df: pd.DataFrame) -> str:
    if df.empty:
        return "합계"
    if "합계" in df.columns:
        return "합계"
    candidates = []
    for c in df.columns:
        if c.lower() in ("total", "sum", "total_amount", "overall", "grand_total"):
            candidates.append(c)
    for c in candidates:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    for c in df.columns:
        if df[c].astype(str).str.contains("PASS", case=False, na=False).any():
            return c
    raise ValueError("Could not detect total column containing PASS/FAIL string.")

TOTAL_COL = detect_total_col(df_final)
TOTAL_COL

'합계'

In [9]:
# =========================
# Cell 8) Base remark (no change events) - FIXED
# =========================

DEFAULT_REMARK_FALLBACK = "Non-PD"  # 이벤트도 없고 remark도 못 찾으면 이걸로

def _clean_remark(v) -> Optional[str]:
    if v is None:
        return None
    s = str(v).strip()
    if s == "" or s.lower() == "nan" or s.lower() == "null":
        return None
    # 표준화
    if s.upper() == "PD":
        return "PD"
    if s.replace(" ", "").replace("_", "").replace("-", "").lower() in ("nonpd", "non-pd", "non_pd"):
        return "Non-PD"
    # 그 외 값이면 그대로 두되, 필요하면 여기서 걸러도 됨
    return s

def get_base_remark_for_station(df_final: pd.DataFrame, station: str) -> Optional[str]:
    s = df_final[df_final["station"] == station].copy()
    if s.empty or "remark" not in s.columns:
        return None

    cleaned = []
    for v in s["remark"].tolist():
        cv = _clean_remark(v)
        if cv is not None:
            cleaned.append(cv)

    vals = sorted(set(cleaned))
    if len(vals) == 0:
        return None
    if len(vals) > 1:
        # station에서 remark가 여러 개면, 실제로는 remark_change가 있어야 정상인데
        # 혹시 데이터가 섞였을 수 있음 -> 일단 첫 번째 사용 + 경고
        print(f"[WARN] station={station} has multiple remark values in final_amount: {vals} -> using first")
    return vals[0]

base_remark = {st: (get_base_remark_for_station(df_final, st) or DEFAULT_REMARK_FALLBACK) for st in STATIONS}
base_remark

[WARN] station=FCT1 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=FCT2 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=FCT3 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=FCT4 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=Vision1 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first
[WARN] station=Vision2 has multiple remark values in final_amount: ['Non-PD', 'PD'] -> using first


{'FCT1': 'Non-PD',
 'FCT2': 'Non-PD',
 'FCT3': 'Non-PD',
 'FCT4': 'Non-PD',
 'Vision1': 'Non-PD',
 'Vision2': 'Non-PD'}

In [10]:
# =========================
# Cell 9) Remark segments (FINAL: 43200 / half-open / boundary=at_time+1s)
# =========================

def _at_time_to_dt(at_time_val) -> datetime:
    """
    at_time(TEXT HH:MI:SS)를 KST datetime으로.
    night shift에서 자정 이후 시간은 날짜 +1.
    """
    at_t = parse_hms_flexible(at_time_val)  # ✅ Cell 3에 정의되어 NameError 해결
    at_dt = datetime.combine(WINDOW_START.date(), at_t, tzinfo=KST)
    if SHIFT_TYPE == "night" and at_dt < WINDOW_START:
        at_dt += timedelta(days=1)
    return at_dt

def build_remark_segments_for_station(df_remark: pd.DataFrame, station: str) -> List[tuple[str, Interval]]:
    """
    half-open [start, end)
    boundary = at_time + 1초
      이전: [cur_start, boundary) -> at_time 포함
      다음: [boundary, ...)
    """
    events = df_remark[df_remark["station"] == station].copy()
    events = events.sort_values("at_time")

    segs: List[tuple[str, Interval]] = []
    cur_start = WINDOW_START

    if events.empty:
        r0 = base_remark.get(station)
        if r0 is None:
            raise ValueError(f"No remark events and no base remark for station={station}")
        return [(r0, Interval(WINDOW_START, WINDOW_END))]

    cur_remark = str(events.iloc[0]["from_remark"])

    for _, row in events.iterrows():
        at_dt = _at_time_to_dt(row["at_time"])
        boundary = at_dt + timedelta(seconds=1)

        # clip to window
        if boundary < WINDOW_START:
            boundary = WINDOW_START
        if boundary > WINDOW_END:
            boundary = WINDOW_END

        if boundary > cur_start:
            segs.append((cur_remark, Interval(cur_start, boundary)))

        cur_start = boundary
        cur_remark = str(row["to_remark"])

        if cur_start >= WINDOW_END:
            break

    if cur_start < WINDOW_END:
        segs.append((cur_remark, Interval(cur_start, WINDOW_END)))

    # 0길이 제거 + 인접 동일 remark 병합
    segs = [(r, iv) for (r, iv) in segs if iv.end > iv.start]

    merged: List[tuple[str, Interval]] = []
    for r, iv in segs:
        if not merged:
            merged.append((r, iv))
            continue
        r_prev, iv_prev = merged[-1]
        if r_prev == r and iv_prev.end == iv.start:
            merged[-1] = (r_prev, Interval(iv_prev.start, iv.end))
        else:
            merged.append((r, iv))

    return merged

segments = {st: build_remark_segments_for_station(df_remark, st) for st in STATIONS}

# 검증: station별 합이 43200초
check_sum = {st: sum(int((iv.end - iv.start).total_seconds()) for _, iv in segments[st]) for st in STATIONS}
print("[CHECK] segment total seconds:", check_sum)

segments

[CHECK] segment total seconds: {'FCT1': 43200, 'FCT2': 43200, 'FCT3': 43200, 'FCT4': 43200, 'Vision1': 43200, 'Vision2': 43200}


{'FCT1': [('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 8, 30, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 10, 55, 49, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 10, 55, 49, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 13, 9, 1, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 13, 9, 1, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 14, 32, 2, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('PD',
   Interval(start=datetime.datetime(2026, 1, 28, 14, 32, 2, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 14, 33, 42, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')))),
  ('Non-PD',
   Interval(start=datetime.datetime(2026, 1, 28, 14, 33, 42, tzinfo=zoneinfo.ZoneInfo(key='Asia/Seoul')), end=datetime.datetime(2026, 1, 28, 14, 3

In [11]:
# =========================
# Cell 10) Planned stop union (skip missing/0 rows)
# =========================

def _is_missing_time(v) -> bool:
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return True
    if s in ("0", "0초", "00", "0000", "000000"):
        return True
    return False

def stops_to_intervals(df_stop: pd.DataFrame, station: Optional[str]) -> List[Interval]:
    d = df_stop.copy()
    if station is not None and "station" in d.columns:
        d = d[d["station"] == station]

    ivs: List[Interval] = []
    for _, row in d.iterrows():
        ft = row.get("from_time", None)
        tt = row.get("to_time", None)

        # ✅ from/to 비어있거나 0이면 실구간 없음 -> 스킵
        if _is_missing_time(ft) or _is_missing_time(tt):
            continue

        ft_t = parse_hms_flexible(ft)
        tt_t = parse_hms_flexible(tt)

        sdt = datetime.combine(WINDOW_START.date(), ft_t, tzinfo=KST)
        edt = datetime.combine(WINDOW_START.date(), tt_t, tzinfo=KST)

        if SHIFT_TYPE == "night":
            if sdt < WINDOW_START:
                sdt += timedelta(days=1)
            if edt < WINDOW_START:
                edt += timedelta(days=1)

        # 뒤집힘 방지
        if edt < sdt:
            sdt, edt = edt, sdt

        # half-open: 길이 0이면 제거
        if edt <= sdt:
            continue

        # window clip (원칙상 밖으로 안 나간다 했지만 방어)
        sdt = max(sdt, WINDOW_START)
        edt = min(edt, WINDOW_END)

        if edt > sdt:
            ivs.append(Interval(sdt, edt))

    return merge_intervals(ivs)

stop_union: Dict[str, List[Interval]] = {}
if "station" in df_stop.columns:
    for st in STATIONS:
        stop_union[st] = stops_to_intervals(df_stop, st)
else:
    common = stops_to_intervals(df_stop, None)
    for st in STATIONS:
        stop_union[st] = common

# 확인: station별 planned stop union 총합(초)
{st: total_seconds(ivs) for st, ivs in stop_union.items()}

{'FCT1': 600,
 'FCT2': 600,
 'FCT3': 600,
 'FCT4': 600,
 'Vision1': 600,
 'Vision2': 600}

In [12]:
# =========================
# Cell 11) planned_stop_sec by station & remark
# =========================

def get_total_planned_time_if_available(df_stop: pd.DataFrame) -> Optional[int]:
    """
    from/to가 없는 행이 많을 수 있으므로, total_planned_time이 있으면 보조로 사용 가능.
    다만 이 값이 "행별 duration"인지 "shift 총합"인지 DB에 따라 다르니,
    여기서는 '이벤트가 아예 없는 station(remark 분해 불가)일 때'만 사용.
    """
    if "total_planned_time" not in df_stop.columns:
        return None
    s = df_stop.dropna(subset=["total_planned_time"])
    if s.empty:
        return None
    try:
        # shift 총합이라고 가정(첫 행) — 필요 시 sum으로 바꿔도 됨
        return int(float(s.iloc[0]["total_planned_time"]))
    except Exception:
        return None

total_planned_time_shift = get_total_planned_time_if_available(df_stop)

planned_sec_by_station_remark: Dict[Tuple[str, str], int] = {}

for st in STATIONS:
    has_events = not df_remark[df_remark["station"] == st].empty

    # init
    for r in REMARKS:
        planned_sec_by_station_remark[(st, r)] = 0

    if not has_events:
        # remark 변화 없으면 base_remark 하나만 43200을 커버하므로 planned도 그 remark에만 넣음
        r0 = base_remark.get(st)
        if r0 is None:
            raise ValueError(f"station={st}: base remark missing")

        if total_planned_time_shift is not None:
            planned_sec_by_station_remark[(st, r0)] = int(total_planned_time_shift)
        else:
            planned_sec_by_station_remark[(st, r0)] = int(total_seconds(stop_union[st]))
        continue

    # events 있는 경우: planned stop union을 remark segments에 overlap 분배
    for r, seg_iv in segments[st]:
        sec = 0
        for ps in stop_union[st]:
            sec += overlap_seconds(ps, seg_iv)
        planned_sec_by_station_remark[(st, r)] += sec

# 확인
planned_sec_by_station_remark

{('FCT1', 'PD'): 251,
 ('FCT1', 'Non-PD'): 349,
 ('FCT2', 'PD'): 266,
 ('FCT2', 'Non-PD'): 334,
 ('FCT3', 'PD'): 600,
 ('FCT3', 'Non-PD'): 0,
 ('FCT4', 'PD'): 600,
 ('FCT4', 'Non-PD'): 0,
 ('Vision1', 'PD'): 219,
 ('Vision1', 'Non-PD'): 381,
 ('Vision2', 'PD'): 98,
 ('Vision2', 'Non-PD'): 502}

In [14]:
# =========================
# Cell 12) Actual PASS by station & remark
#   FIX:
#     station 컬럼이 있는 테이블을 사용해야 함
#     - i_daily_report.a_station_day_daily_final_amount  (day)
#     - i_daily_report.a_station_night_daily_final_amount (night)
# =========================
import pandas as pd
import re
from sqlalchemy import text

# 1) 테이블 선택 (station이 있는 버전)
T_DAY_STATION  = "a_station_day_daily_final_amount"
T_NIGHT_STATION = "a_station_night_daily_final_amount"
station_final_table = T_DAY_STATION if SHIFT_TYPE == "day" else T_NIGHT_STATION
station_final_fqn = f'"{SAVE_SCHEMA}"."{station_final_table}"'

# 2) PASS/FAIL 파서
PASS_RE = re.compile(r"PASS\s*[:=]\s*(\d+)", re.IGNORECASE)
FAIL_RE = re.compile(r"FAIL\s*[:=]\s*(\d+)", re.IGNORECASE)

def parse_pass_from_sum_text(x) -> int:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    m = PASS_RE.search(s)
    return int(m.group(1)) if m else 0

def parse_total_from_sum_text(x) -> int:
    """PASS+FAIL = Total"""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return 0
    s = str(x)
    mp = PASS_RE.search(s)
    mf = FAIL_RE.search(s)
    p = int(mp.group(1)) if mp else 0
    f = int(mf.group(1)) if mf else 0
    return p + f

# 3) 합계 컬럼 감지
cols_station_final = get_columns(engine, SAVE_SCHEMA, station_final_table)

def pick_first(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

sum_col = pick_first(cols_station_final, ["합계", "total", "Total", "SUM", "sum"])
if sum_col is None:
    raise ValueError(f"{station_final_table}에서 합계 컬럼을 못 찾았습니다. cols={cols_station_final}")

# 필수 컬럼 체크
need_cols = ["prod_day", "shift_type", "station", "remark"]
for c in need_cols:
    if c not in cols_station_final:
        raise ValueError(f"{station_final_table}에 필수 컬럼 '{c}' 없음. cols={cols_station_final}")

# 4) 로드
sql = text(f"""
    SELECT prod_day, shift_type, station, remark, "{sum_col}" AS sum_text
    FROM {station_final_fqn}
    WHERE prod_day = :prod_day
      AND shift_type = :shift_type
      AND station = ANY(:stations)
      AND remark  = ANY(:remarks)
""")

with engine.begin() as conn:
    df_station_final = pd.read_sql(sql, conn, params={
        "prod_day": PROD_DAY,
        "shift_type": SHIFT_TYPE,
        "stations": list(STATIONS),   # ["FCT1"~"FCT4","Vision1","Vision2"]
        "remarks": list(REMARKS),     # ["PD","Non-PD"]
    })

if df_station_final.empty:
    raise ValueError(f"{station_final_table} returned 0 rows. Check prod_day/shift_type/stations/remarks.")

# 5) 파싱
df_station_final["actual_pass_qty"]  = df_station_final["sum_text"].apply(parse_pass_from_sum_text).astype(int)
df_station_final["actual_total_qty"] = df_station_final["sum_text"].apply(parse_total_from_sum_text).astype(int)

# 6) station×remark 합산
actual_by_station_remark = (
    df_station_final
    .groupby(["station", "remark"], as_index=False)
    .agg(
        actual_pass_qty=("actual_pass_qty", "sum"),
        actual_total_qty=("actual_total_qty", "sum"),
    )
    .sort_values(["station", "remark"])
    .reset_index(drop=True)
)

actual_by_station_remark

,station,remark,actual_pass_qty,actual_total_qty
0,FCT1,Non-PD,204,204
1,FCT1,PD,566,566
2,FCT2,Non-PD,221,221
3,FCT2,PD,606,606
4,FCT3,Non-PD,206,206
5,FCT3,PD,583,585
6,FCT4,Non-PD,223,224
7,FCT4,PD,594,598
8,Vision1,Non-PD,418,418
9,Vision1,PD,1167,1168


In [15]:
# =========================
# Cell 13) Ideal CT load (UPDATED: Vision=ct_eq MIN / FCT=del_out_op_ct_av MIN)
# =========================
import pandas as pd
from sqlalchemy import text

SIDE_TO_STATION = {"left": "Vision1", "right": "Vision2"}

def pick_col(cols: List[str], candidates: List[str], required=True) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    if required:
        raise ValueError(f"Missing columns, tried={candidates}, have={cols}")
    return None

# ---- Vision Ideal CT: e1_FCT_ct.fct_whole_op_ct 의 ct_eq 최소값 (remark별, side별) ----
vis_col_side   = pick_col(cols_ideal_vision, ["station"])   # left/right 저장
vis_col_remark = pick_col(cols_ideal_vision, ["remark"])
vis_col_ct     = pick_col(cols_ideal_vision, ["ct_eq"])     # ✅ 요구사항: ct_eq MIN

# (옵션) 날짜/shift 컬럼이 있으면 IDEAL_BASE_PROD_DAY로 필터
vis_col_prod  = pick_col(cols_ideal_vision, ["prod_day","end_day","day","work_day"], required=False)
vis_col_shift = pick_col(cols_ideal_vision, ["shift_type","shift"], required=False)

where = []
params = {}

if vis_col_prod is not None:
    where.append(f"{vis_col_prod} = :base_prod_day")
    params["base_prod_day"] = IDEAL_BASE_PROD_DAY
if vis_col_shift is not None:
    where.append(f"{vis_col_shift} = :shift_type")
    params["shift_type"] = SHIFT_TYPE

# side/remark 제한(안전)
where.append(f"{vis_col_side} = ANY(:sides)")
params["sides"] = ["left", "right"]
where.append(f"{vis_col_remark} = ANY(:remarks)")
params["remarks"] = list(REMARKS)

where_sql = "WHERE " + " AND ".join(where)

sql_vision = text(f"""
    SELECT {vis_col_side}   AS side,
           {vis_col_remark} AS remark,
           MIN({vis_col_ct}) AS ideal_ct_sec
    FROM {ideal_vision_fqn}
    {where_sql}
    GROUP BY {vis_col_side}, {vis_col_remark}
""")

with engine.begin() as conn:
    df_vision = pd.read_sql(sql_vision, conn, params=params)

df_vision["side"] = df_vision["side"].astype(str).str.strip().str.lower()
df_vision["station"] = df_vision["side"].map(SIDE_TO_STATION)

# 매핑 실패 방어
bad = df_vision[df_vision["station"].isna()][["side","remark","ideal_ct_sec"]]
if not bad.empty:
    print("[WARN] Unexpected side values in fct_whole_op_ct.station (expected left/right):")
    display(bad)

df_vision = df_vision.dropna(subset=["station"])[["station","remark","ideal_ct_sec"]].copy()

# ---- FCT Ideal CT: e1_FCT_ct.fct_op_ct 의 del_out_op_ct_av 최소값 (station별, remark별) ----
if "del_out_op_ct_av" not in cols_ideal_fct:
    raise ValueError(
        f"FCT ideal 테이블({IDEAL_SCHEMA_FCT}.{IDEAL_TABLE_FCT})에 del_out_op_ct_av 컬럼이 없습니다: {cols_ideal_fct}"
    )

fct_col_station = pick_col(cols_ideal_fct, ["station"])
fct_col_remark  = pick_col(cols_ideal_fct, ["remark"])

sql_fct = text(f"""
    SELECT {fct_col_station} AS station,
           {fct_col_remark}  AS remark,
           MIN(del_out_op_ct_av) AS ideal_ct_sec
    FROM {ideal_fct_fqn}
    WHERE {fct_col_station} = ANY(:stations)
      AND {fct_col_remark}  = ANY(:remarks)
    GROUP BY {fct_col_station}, {fct_col_remark}
""")

with engine.begin() as conn:
    df_fct = pd.read_sql(sql_fct, conn, params={
        "stations": ["FCT1","FCT2","FCT3","FCT4"],
        "remarks": list(REMARKS)
    })

df_fct = df_fct[["station","remark","ideal_ct_sec"]].copy()

# ---- merge ----
df_ideal = pd.concat([df_vision, df_fct], ignore_index=True)

# (검증) Vision 4개 조합이 다 있는지 확인
need_vis = {( "Vision1","PD"),("Vision1","Non-PD"),("Vision2","PD"),("Vision2","Non-PD")}
have_vis = set(tuple(x) for x in df_ideal[df_ideal["station"].isin(["Vision1","Vision2"])][["station","remark"]].values.tolist())
miss = sorted(list(need_vis - have_vis))
if miss:
    print("[WARN] Missing Vision ideal_ct combos:", miss)

df_ideal

,station,remark,ideal_ct_sec
0,Vision2,PD,17.67
1,Vision1,PD,18.35
2,Vision2,Non-PD,15.44
3,Vision1,Non-PD,15.18
4,FCT4,Non-PD,30.77
5,FCT2,PD,36.59
6,FCT3,PD,35.54
7,FCT4,PD,35.14
8,FCT1,Non-PD,30.57
9,FCT3,Non-PD,31.01


In [16]:
# =========================
# Cell 14) OEE (Actual / Ideal) final
# =========================

def remark_window_seconds_for(station: str, remark: str) -> int:
    sec = 0
    for r, iv in segments[station]:
        if r == remark:
            sec += int((iv.end - iv.start).total_seconds())
    return sec

# 검증: station별 PD+Non-PD 합이 43200
sum_by_station = {st: sum(remark_window_seconds_for(st, r) for r in REMARKS) for st in STATIONS}
print("[CHECK] remark_window_sec sum:", sum_by_station)

rows = []
for st in STATIONS:
    for r in REMARKS:
        remark_window_sec = remark_window_seconds_for(st, r)
        if remark_window_sec <= 0:
            continue

        planned_sec = int(planned_sec_by_station_remark.get((st, r), 0))
        if planned_sec > remark_window_sec:
            print(f"[WARN] planned_sec > remark_window_sec: {st},{r} {planned_sec}>{remark_window_sec} -> cap")
            planned_sec = remark_window_sec

        effective_sec = remark_window_sec - planned_sec

        ideal_row = df_ideal[(df_ideal["station"] == st) & (df_ideal["remark"] == r)]
        ideal_ct = float(ideal_row.iloc[0]["ideal_ct_sec"]) if not ideal_row.empty else None

        act_row = actual_by_station_remark[
            (actual_by_station_remark["station"] == st) &
            (actual_by_station_remark["remark"] == r)
        ]
        actual = int(act_row.iloc[0]["actual_pass_qty"]) if not act_row.empty else 0

        ideal_qty = (effective_sec / ideal_ct) if (ideal_ct and ideal_ct > 0) else None

        # ✅ 너가 말하는 OEE = Actual / Ideal
        oee = (actual / ideal_qty) if (ideal_qty is not None and ideal_qty > 0) else None

        rows.append({
            "prod_day": PROD_DAY,
            "shift_type": SHIFT_TYPE,
            "station": st,
            "remark": r,
            "remark_window_sec": remark_window_sec,
            "planned_stop_sec": planned_sec,
            "effective_sec": effective_sec,
            "ideal_ct_sec": ideal_ct,
            "ideal_qty": ideal_qty,
            "actual_pass_qty": actual,
            "oee_actual_over_ideal": oee,
            "updated_at": datetime.now(tz=KST),
        })

df_oee = pd.DataFrame(rows).sort_values(["station", "remark"]).reset_index(drop=True)
df_oee

[CHECK] remark_window_sec sum: {'FCT1': 43200, 'FCT2': 43200, 'FCT3': 43200, 'FCT4': 43200, 'Vision1': 43200, 'Vision2': 43200}


,prod_day,shift_type,station,remark,remark_window_sec,planned_stop_sec,effective_sec,ideal_ct_sec,ideal_qty,actual_pass_qty,oee_actual_over_ideal,updated_at
0,20260128,day,FCT1,Non-PD,14906,349,14557,30.57,476.185803,204,0.428404,2026-02-03 13:51:24.198475+09:00
1,20260128,day,FCT1,PD,28294,251,28043,36.73,763.490335,566,0.741332,2026-02-03 13:51:24.197296+09:00
2,20260128,day,FCT2,Non-PD,15038,334,14704,30.16,487.533156,221,0.453303,2026-02-03 13:51:24.201060+09:00
3,20260128,day,FCT2,PD,28162,266,27896,36.59,762.394097,606,0.794864,2026-02-03 13:51:24.199912+09:00
4,20260128,day,FCT3,Non-PD,14237,0,14237,31.01,459.109965,206,0.448694,2026-02-03 13:51:24.203280+09:00
5,20260128,day,FCT3,PD,28963,600,28363,35.54,798.058526,583,0.730523,2026-02-03 13:51:24.202181+09:00
6,20260128,day,FCT4,Non-PD,14336,0,14336,30.77,465.908352,223,0.478635,2026-02-03 13:51:24.205529+09:00
7,20260128,day,FCT4,PD,28864,600,28264,35.14,804.325555,594,0.738507,2026-02-03 13:51:24.204300+09:00
8,20260128,day,Vision1,Non-PD,8781,381,8400,15.18,553.359684,418,0.755386,2026-02-03 13:51:24.207979+09:00
9,20260128,day,Vision1,PD,34419,219,34200,18.35,1863.760218,1167,0.626154,2026-02-03 13:51:24.206758+09:00


In [23]:
# =========================
# Cell 15) OEE Outputs
#   1) df_oee_total  : TOTAL OEE (Line1+Line2, Vision-based)
#   2) df_oee_by_line: Line1/Line2 OEE (Vision1/Vision2 기준)
#   3) df_oee_by_station: Station OEE (FCT1~4 + Vision1~2)
# =========================
import re
import numpy as np
import pandas as pd

# ---------- helpers ----------
def parse_korean_duration_to_sec(v) -> int:
    if v is None:
        return 0
    s = str(v).strip()
    if s == "" or s.lower() in ("nan", "none", "null"):
        return 0
    h = m = sec = 0
    mh = re.search(r"(\d+)\s*시간", s)
    mm = re.search(r"(\d+)\s*분", s)
    ms = re.search(r"(\d+)\s*초", s)
    if mh: h = int(mh.group(1))
    if mm: m = int(mm.group(1))
    if ms: sec = int(ms.group(1))
    return h * 3600 + m * 60 + sec

def parse_quality_text(v) -> tuple[int, int, float]:
    """
    예) "PASS: 777, total: 835, PASS_pct:93.05"
    -> (pass=777, total=835, Q=0.9305)
    """
    if v is None:
        return 0, 0, np.nan
    s = str(v)
    mp   = re.search(r"PASS\s*:\s*(\d+)", s, re.IGNORECASE)
    mt   = re.search(r"total\s*:\s*(\d+)", s, re.IGNORECASE)
    mpct = re.search(r"PASS_pct\s*:\s*([0-9]+(?:\.[0-9]+)?)", s, re.IGNORECASE)
    p = int(mp.group(1)) if mp else 0
    t = int(mt.group(1)) if mt else 0
    q = float(mpct.group(1))/100.0 if mpct else (p/t if t > 0 else np.nan)
    return p, t, q

def _safe_div(num, den):
    den = float(den)
    if np.isnan(den) or den <= 0:
        return np.nan
    return float(num) / den

def to_pct_str(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return f"{x*100:.2f}%"

def ideal_ct_for(station: str, remark: str) -> float | None:
    s = df_ideal[(df_ideal["station"] == station) & (df_ideal["remark"] == remark)]
    if s.empty:
        return None
    try:
        v = float(s.iloc[0]["ideal_ct_sec"])
        return v if v > 0 else None
    except Exception:
        return None

def remark_window_seconds_for(station: str, remark: str) -> int:
    sec = 0
    for rr, iv in segments[station]:
        if str(rr) == remark:
            sec += int((iv.end - iv.start).total_seconds())
    return sec

# ---------- required inputs check ----------
for need_name in ["df_non", "df_q", "segments", "planned_sec_by_station_remark", "df_ideal"]:
    if need_name not in globals():
        raise ValueError(f"Missing required object: {need_name} (check previous cells)")

# ---------- load non_time + quality ----------
if df_non.empty:
    raise ValueError("non_time table returned 0 rows. Check prod_day/shift_type.")
non_row = df_non.iloc[0].to_dict()

if df_q.empty:
    raise ValueError("quality table returned 0 rows. Check prod_day/shift_type.")
q_row = df_q.iloc[0].to_dict()

def non_time_sec(station: str) -> int:
    col = f"비가동 {station}"
    if col not in df_non.columns:
        raise ValueError(f"non_time column not found: {col}")
    return parse_korean_duration_to_sec(non_row.get(col))

def quality_triplet(station: str) -> dict:
    if station not in df_q.columns:
        raise ValueError(f"quality column not found: {station}")
    p, t, q = parse_quality_text(q_row.get(station))
    return {"pass": p, "total": t, "Q": q}

# ---------- stations / lines ----------
STATIONS_ALL = ["FCT1","FCT2","FCT3","FCT4","Vision1","Vision2"]
LINES = [
    {"line": "left",  "vision": "Vision1", "fcts": ["FCT1", "FCT2"]},
    {"line": "right", "vision": "Vision2", "fcts": ["FCT3", "FCT4"]},
]


# =========================================================
# Part A) Station OEE (FCT1~4 + Vision1~2)
# =========================================================
station_rows = []
station_intermediate = {}

for st in STATIONS_ALL:
    # planned stop (remark별로 분배된 값 합산)
    planned_total = sum(int(planned_sec_by_station_remark.get((st, r), 0)) for r in REMARKS)

    # Planned Production Time (PPT)
    PPT = max(WINDOW_SECONDS - planned_total, 0)

    # Unplanned stop (non_time)
    unplanned = int(non_time_sec(st))

    # Run Time
    run = max(PPT - unplanned, 0)

    # Availability
    A = _safe_div(run, PPT)

    # Ideal total qty (remark-aware)
    ideal_total_qty = 0.0
    for r in REMARKS:
        rw = remark_window_seconds_for(st, r)
        if rw <= 0:
            continue
        ps_r = min(int(planned_sec_by_station_remark.get((st, r), 0)), rw)
        PPT_r = max(rw - ps_r, 0)
        ct = ideal_ct_for(st, r)
        if ct is None:
            continue
        ideal_total_qty += (PPT_r / ct)

    mix_ct = (PPT / ideal_total_qty) if (PPT > 0 and ideal_total_qty > 0) else np.nan

    # TotalCount & Quality (quality table 기반)
    qt = quality_triplet(st)
    total_cnt = int(qt["total"])
    good_cnt  = int(qt["pass"])
    Q = float(qt["Q"])

    # Performance
    P = _safe_div(mix_ct * total_cnt, run) if run > 0 else np.nan

    # OEE
    OEE = (A * P * Q) if (not np.isnan(A) and not np.isnan(P) and not np.isnan(Q)) else np.nan

    station_rows.append({
        "prod_day": PROD_DAY,
        "shift_type": SHIFT_TYPE,
        "station": st,
        "oee_station": to_pct_str(OEE),
    })

    station_intermediate[st] = dict(PPT=PPT, run=run, ideal_qty=ideal_total_qty, total_cnt=total_cnt, good_cnt=good_cnt)

df_oee_by_station = pd.DataFrame(station_rows)

# =========================================================
# Part B) Line OEE (Vision-based)
#   - Line1 uses Vision1
#   - Line2 uses Vision2
# =========================================================
line_rows = []
line_intermediate = {}

for L in LINES:
    line_name = L["line"]
    vis = L["vision"]

    v = station_intermediate[vis]  # Vision station에서 이미 계산한 값 재사용

    PPT = v["PPT"]
    run = v["run"]
    ideal_qty = v["ideal_qty"]
    total_cnt = v["total_cnt"]
    good_cnt  = v["good_cnt"]

    A = _safe_div(run, PPT)
    mix_ct = (PPT / ideal_qty) if (PPT > 0 and ideal_qty > 0) else np.nan
    P = _safe_div(mix_ct * total_cnt, run) if run > 0 else np.nan
    Q = _safe_div(good_cnt, total_cnt)

    OEE = (A * P * Q) if (not np.isnan(A) and not np.isnan(P) and not np.isnan(Q)) else np.nan

    line_rows.append({
        "prod_day": PROD_DAY,
        "shift_type": SHIFT_TYPE,
        "line": line_name,
        "oee_line": to_pct_str(OEE),
    })

    line_intermediate[line_name] = dict(PPT=PPT, run=run, ideal_qty=ideal_qty, total_cnt=total_cnt, good_cnt=good_cnt)

df_oee_by_line = pd.DataFrame(line_rows)

# =========================================================
# Part C) TOTAL OEE (Line1 + Line2 합산 APQ)
# =========================================================
PPT_tot   = sum(v["PPT"] for v in line_intermediate.values())
run_tot   = sum(v["run"] for v in line_intermediate.values())
ideal_tot = sum(v["ideal_qty"] for v in line_intermediate.values())
total_cnt_tot = int(sum(v["total_cnt"] for v in line_intermediate.values()))
good_cnt_tot  = int(sum(v["good_cnt"]  for v in line_intermediate.values()))

Q_tot = _safe_div(good_cnt_tot, total_cnt_tot)
A_tot = _safe_div(run_tot, PPT_tot)
mix_ct_tot = (PPT_tot / ideal_tot) if (PPT_tot > 0 and ideal_tot > 0) else np.nan
P_tot = _safe_div(mix_ct_tot * total_cnt_tot, run_tot) if run_tot > 0 else np.nan

OEE_tot = (A_tot * P_tot * Q_tot) if (not np.isnan(A_tot) and not np.isnan(P_tot) and not np.isnan(Q_tot)) else np.nan

df_oee_total = pd.DataFrame([{
    "prod_day": PROD_DAY,
    "shift_type": SHIFT_TYPE,
    "전체 OEE": to_pct_str(OEE_tot),
}])

# ---------- outputs ----------
display(df_oee_total)
display(df_oee_by_line)
display(df_oee_by_station)

,prod_day,shift_type,전체 OEE
0,20260128,day,65.47%


,prod_day,shift_type,line,oee_line
0,20260128,day,left,65.62%
1,20260128,day,right,65.34%


,prod_day,shift_type,station,oee_station
0,20260128,day,FCT1,62.11%
1,20260128,day,FCT2,66.16%
2,20260128,day,FCT3,62.76%
3,20260128,day,FCT4,64.32%
4,20260128,day,Vision1,65.61%
5,20260128,day,Vision2,65.33%


In [26]:
# =========================
# Cell 16) Save OEE DataFrames to DB (CREATE IF NOT EXISTS + UPSERT) - KOREAN COL NAMES
# =========================
from sqlalchemy import text
import pandas as pd

SAVE_SCHEMA = "i_daily_report"

# --- target tables by shift ---
T_TOTAL_DAY   = "k_total_oee_day_daily"
T_TOTAL_NIGHT = "k_total_oee_night_daily"

T_LINE_DAY    = "k_line_oee_day_daily"
T_LINE_NIGHT  = "k_line_oee_night_daily"

T_ST_DAY      = "k_station_oee_day_daily"
T_ST_NIGHT    = "k_station_oee_night_daily"

t_total   = T_TOTAL_DAY   if SHIFT_TYPE == "day" else T_TOTAL_NIGHT
t_line    = T_LINE_DAY    if SHIFT_TYPE == "day" else T_LINE_NIGHT
t_station = T_ST_DAY      if SHIFT_TYPE == "day" else T_ST_NIGHT

fqn_total   = f'"{SAVE_SCHEMA}"."{t_total}"'
fqn_line    = f'"{SAVE_SCHEMA}"."{t_line}"'
fqn_station = f'"{SAVE_SCHEMA}"."{t_station}"'

# --- sanity checks ---
for name in ["df_oee_total", "df_oee_by_line", "df_oee_by_station"]:
    if name not in globals():
        raise ValueError(f"Missing dataframe: {name}. Run Cell 15 first.")

if df_oee_total.empty:
    raise ValueError("df_oee_total is empty")
if df_oee_by_line.empty:
    raise ValueError("df_oee_by_line is empty")
if df_oee_by_station.empty:
    raise ValueError("df_oee_by_station is empty")

# --- normalize dataframe columns for DB ---
# df_oee_total: prod_day, shift_type, "전체 OEE" (또는 oee_total)
df_total_db = df_oee_total.copy()
if "전체 OEE" not in df_total_db.columns:
    if "oee_total" in df_total_db.columns:
        df_total_db = df_total_db.rename(columns={"oee_total": "전체 OEE"})
    else:
        raise ValueError("df_oee_total must have column '전체 OEE' (or 'oee_total').")
df_total_db = df_total_db[["prod_day", "shift_type", "전체 OEE"]].copy()

# df_oee_by_line: prod_day, shift_type, line, "Line별 OEE" (또는 oee_line)
df_line_db = df_oee_by_line.copy()
if "Line별 OEE" not in df_line_db.columns:
    if "oee_line" in df_line_db.columns:
        df_line_db = df_line_db.rename(columns={"oee_line": "Line별 OEE"})
    else:
        raise ValueError("df_oee_by_line must have column 'Line별 OEE' (or 'oee_line').")
need_line = ["prod_day", "shift_type", "line", "Line별 OEE"]
missing = [c for c in need_line if c not in df_line_db.columns]
if missing:
    raise ValueError(f"df_oee_by_line missing columns: {missing}")
df_line_db = df_line_db[need_line].copy()

# df_oee_by_station: prod_day, shift_type, station, "Station별 OEE" (또는 oee_station)
df_station_db = df_oee_by_station.copy()
if "Station별 OEE" not in df_station_db.columns:
    if "oee_station" in df_station_db.columns:
        df_station_db = df_station_db.rename(columns={"oee_station": "Station별 OEE"})
    else:
        raise ValueError("df_oee_by_station must have column 'Station별 OEE' (or 'oee_station').")
need_st = ["prod_day", "shift_type", "station", "Station별 OEE"]
missing = [c for c in need_st if c not in df_station_db.columns]
if missing:
    raise ValueError(f"df_oee_by_station missing columns: {missing}")
df_station_db = df_station_db[need_st].copy()

# --- create schema/table if not exist (PK 포함) ---
ddl_schema = text(f'CREATE SCHEMA IF NOT EXISTS "{SAVE_SCHEMA}";')

ddl_total = text(f"""
CREATE TABLE IF NOT EXISTS {fqn_total} (
    prod_day   text NOT NULL,
    shift_type text NOT NULL,
    "전체 OEE"  text,
    updated_at timestamptz NOT NULL DEFAULT now(),
    PRIMARY KEY (prod_day, shift_type)
);
""")

ddl_line = text(f"""
CREATE TABLE IF NOT EXISTS {fqn_line} (
    prod_day     text NOT NULL,
    shift_type   text NOT NULL,
    line         text NOT NULL,
    "Line별 OEE" text,
    updated_at   timestamptz NOT NULL DEFAULT now(),
    PRIMARY KEY (prod_day, shift_type, line)
);
""")

ddl_station = text(f"""
CREATE TABLE IF NOT EXISTS {fqn_station} (
    prod_day        text NOT NULL,
    shift_type      text NOT NULL,
    station         text NOT NULL,
    "Station별 OEE" text,
    updated_at      timestamptz NOT NULL DEFAULT now(),
    PRIMARY KEY (prod_day, shift_type, station)
);
""")

# --- upsert helpers (KOREAN COLS) ---
def upsert_total(conn, df: pd.DataFrame):
    sql = text(f"""
        INSERT INTO {fqn_total} (prod_day, shift_type, "전체 OEE", updated_at)
        VALUES (:prod_day, :shift_type, :전체_OEE, now())
        ON CONFLICT (prod_day, shift_type)
        DO UPDATE SET
            "전체 OEE" = EXCLUDED."전체 OEE",
            updated_at = now()
    """)
    payload = df.rename(columns={"전체 OEE": "전체_OEE"}).to_dict(orient="records")
    conn.execute(sql, payload)

def upsert_line(conn, df: pd.DataFrame):
    sql = text(f"""
        INSERT INTO {fqn_line} (prod_day, shift_type, line, "Line별 OEE", updated_at)
        VALUES (:prod_day, :shift_type, :line, :Line별_OEE, now())
        ON CONFLICT (prod_day, shift_type, line)
        DO UPDATE SET
            "Line별 OEE" = EXCLUDED."Line별 OEE",
            updated_at = now()
    """)
    payload = df.rename(columns={"Line별 OEE": "Line별_OEE"}).to_dict(orient="records")
    conn.execute(sql, payload)

def upsert_station(conn, df: pd.DataFrame):
    sql = text(f"""
        INSERT INTO {fqn_station} (prod_day, shift_type, station, "Station별 OEE", updated_at)
        VALUES (:prod_day, :shift_type, :station, :Station별_OEE, now())
        ON CONFLICT (prod_day, shift_type, station)
        DO UPDATE SET
            "Station별 OEE" = EXCLUDED."Station별 OEE",
            updated_at = now()
    """)
    payload = df.rename(columns={"Station별 OEE": "Station별_OEE"}).to_dict(orient="records")
    conn.execute(sql, payload)

# --- run ---
with engine.begin() as conn:
    conn.execute(ddl_schema)

    # 테이블 보장
    conn.execute(ddl_total)
    conn.execute(ddl_line)
    conn.execute(ddl_station)

    # UPSERT 저장
    upsert_total(conn, df_total_db)
    upsert_line(conn, df_line_db)
    upsert_station(conn, df_station_db)

print("[OK] Saved OEE results (Korean column names):")
print(" -", fqn_total,   '("전체 OEE")')
print(" -", fqn_line,    '("Line별 OEE")')
print(" -", fqn_station, '("Station별 OEE")')

[OK] Saved OEE results (Korean column names):
 - "i_daily_report"."k_total_oee_day_daily" ("전체 OEE")
 - "i_daily_report"."k_line_oee_day_daily" ("Line별 OEE")
 - "i_daily_report"."k_station_oee_day_daily" ("Station별 OEE")
